In [1]:
# ===============================
# Common Setup
# ===============================
import torch
from fastai.text.all import *

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_DIR = "/content/drive/MyDrive/Sentiment_Analysis_Project"
MODEL_DIR = f"{BASE_DIR}/models"

# ensure models folder exists
import os
os.makedirs(MODEL_DIR, exist_ok=True)


In [2]:
# Load IMDb dataset using fastai
path = untar_data(URLs.IMDB)

# Create DataLoaders
dls = TextDataLoaders.from_folder(
    path,
    valid='test',
    is_lm=False,          # classification task
    bs=64
)

dls.show_batch(max_n=2)


,text,category
0,"xxbos xxmaj match 1 : xxmaj tag xxmaj team xxmaj table xxmaj match xxmaj bubba xxmaj ray and xxmaj spike xxmaj dudley vs xxmaj eddie xxmaj guerrero and xxmaj chris xxmaj benoit xxmaj bubba xxmaj ray and xxmaj spike xxmaj dudley started things off with a xxmaj tag xxmaj team xxmaj table xxmaj match against xxmaj eddie xxmaj guerrero and xxmaj chris xxmaj benoit . xxmaj according to the rules of the match , both opponents have to go through tables in order to get the win . xxmaj benoit and xxmaj guerrero heated up early on by taking turns hammering first xxmaj spike and then xxmaj bubba xxmaj ray . a xxmaj german xxunk by xxmaj benoit to xxmaj bubba took the wind out of the xxmaj dudley brother . xxmaj spike tried to help his brother , but the referee restrained him while xxmaj benoit and xxmaj guerrero",pos
1,"xxbos xxmaj heavy - handed moralism . xxmaj writers using characters as mouthpieces to speak for themselves . xxmaj predictable , plodding plot points ( say that five times fast ) . a child 's imitation of xxmaj britney xxmaj spears . xxmaj this film has all the earmarks of a xxmaj lifetime xxmaj special reject . \n\n i honestly believe that xxmaj jesus xxmaj xxunk and xxmaj julia xxmaj xxunk set out to create a thought - provoking , emotional film on a tough subject , exploring the idea that things are not always black and white , that one who is a criminal by definition is not necessarily a bad human being , and that there can be extenuating circumstances , especially when one puts the well - being of a child first . xxmaj however , their earnestness ends up being channeled into preachy dialogue and trite",neg


In [3]:
# Create AWD-LSTM learner
learn = text_classifier_learner(
    dls,
    AWD_LSTM,
    drop_mult=0.5,
    metrics=accuracy
)

# Fine-tune using ULMFiT
learn.fine_tune(2, 1e-2)


epoch,train_loss,valid_loss,accuracy,time
0,0.468428,0.403735,0.816600,03:13


epoch,train_loss,valid_loss,accuracy,time
0,0.298983,0.235823,0.908400,07:25


epoch,train_loss,valid_loss,accuracy,time
0,0.298983,0.235823,0.908400,07:25
1,0.228000,0.202394,0.921040,07:30


In [4]:
# ===============================
# AWD-LSTM FINAL ONE-CELL FIX
# ===============================
from fastai.text.all import *
import os

BASE_DIR = "/content/drive/MyDrive/Sentiment_Analysis_Project"
os.makedirs(f"{BASE_DIR}/models", exist_ok=True)

# Load IMDb dataset
path = untar_data(URLs.IMDB)

dls = TextDataLoaders.from_folder(
    path,
    valid='test',
    is_lm=False,
    bs=64
)

# Recreate learner
learn = text_classifier_learner(
    dls,
    AWD_LSTM,
    drop_mult=0.5,
    metrics=accuracy
)

# Try loading existing model
try:
    learn.load("awd_lstm_ulmfit")
    print("✅ Existing AWD-LSTM model loaded")
except:
    print("⚠️ No saved model found — training 1 epoch")
    learn.freeze()
    learn.fit_one_cycle(1, 1e-2)
    learn.save("awd_lstm_ulmfit")
    print("✅ AWD-LSTM model trained & saved")

# Evaluate
val_loss, val_acc = learn.validate()
print(f"📊 Validation Accuracy: {val_acc:.4f}")



⚠️ No saved model found — training 1 epoch


epoch,train_loss,valid_loss,accuracy,time
0,0.445531,0.369810,0.838960,03:32


✅ AWD-LSTM model trained & saved


📊 Validation Accuracy: 0.8390
